In [92]:
from rdkit import Chem
from rdkit.Chem.Draw import IPythonConsole
from yaml import warnings

IPythonConsole.ipython_useSVG=True
import pandas as pd
import numpy as np
import pandas as pd
#import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.pyplot as plt
import numpy as np
from rdkit import Chem
from rdkit.Chem import Descriptors
import requests
import gzip
import pandas as pd
from io import BytesIO
import requests
from pathlib import Path
from Bio.PDB import PDBList
import os
import shutil
import Bio.PDB
from github import Github
import os
import requests
from pathlib import Path
import pymol
from pymol import cmd
import sys
import time
#import py3Dmol
from IPython.display import display, Image
import tqdm
from tqdm.auto import tqdm
import os
import shutil
from concurrent.futures import ThreadPoolExecutor, as_completed
from openbabel import pybel
import pickle
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from openbabel import pybel
#import pybel
from openbabel import openbabel as ob
import pickle
import subprocess
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import pandas as pd
import json
import gemmi
import io
from rdkit import Chem
from rdkit.Chem.Draw import IPythonConsole
IPythonConsole.ipython_useSVG=True
import sys
import click
from rdkit import Chem
from rdkit.Chem import rdDistGeom
from rdkit.Chem import AllChem
import requests
from openbabel import pybel
import numpy as np
import rdkit
from rdkit import Chem
from rdkit.Chem import Draw
from rdkit.Chem.Draw import IPythonConsole
from rdkit.Chem import Descriptors
from rdkit.Chem import AllChem
from rdkit import DataStructs
from rdkit.Chem import PandasTools
import numpy as np
import pandas as pd
import openbabel
from openbabel import pybel

In [12]:
#! pip install gemmi

In [80]:
structures_dir = Path('./structures')

### All structures were prealigned with TMAlign (python script)

In [ ]:
skip_list = {'HOH', 'CL', 'SO4', 'PO4', 'EDO', 'PEG'}
all_data = []
mon_names = set()

for cif_file in structures_dir.glob("*_out:A.cif"):
    try:
        protein = gemmi.read_structure(str(cif_file.resolve()))
        mon_names.update(protein[0].get_all_residue_names())
        for model in protein:
            for chain in model:
                for residue in chain:
                    if residue.entity_type == gemmi.EntityType.NonPolymer:
                        for atom in residue:
                        # Append a dictionary for each atom
                            all_data.append({
                                'protein': protein.name,
                                'resname': residue.name,
                                'atom_name': atom.name,
                                'element': atom.element.name,
                                'weight': atom.element.weight,
                                'x': atom.pos.x,
                                'y': atom.pos.y,
                                'z': atom.pos.z
                            })
    except Exception as e:
        print(f"Skipping {cif_file} due to error: {e}")

In [94]:
df = pd.DataFrame(all_data)

# All ligands with its coordinates

In [125]:
df[df['protein'] == '6ZIR']

,protein,resname,atom_name,element,weight,x,y,z,wx,wy,wz
133,6ZIR,MG,MG,Mg,24.305000,19.690,39.138,-0.939,478.565450,951.249090,-22.822395
134,6ZIR,MG,MG,Mg,24.305000,7.838,29.739,-22.512,190.502590,722.806395,-547.154160
135,6ZIR,GDP,PB,P,30.973761,17.934,38.566,-3.558,555.483430,1194.534067,-110.204642
136,6ZIR,GDP,O1B,O,15.999400,19.247,38.134,-4.128,307.940452,610.121120,-66.045523
137,6ZIR,GDP,O2B,O,15.999400,17.973,39.034,-2.119,287.557216,624.520580,-33.902729
...,...,...,...,...,...,...,...,...,...,...,...
206,6ZIR,EZZ,N13,N,14.006700,16.320,33.625,12.762,228.589344,470.975288,178.753505
207,6ZIR,EZZ,N21,N,14.006700,17.506,34.243,15.539,245.201290,479.631428,217.650111
208,6ZIR,EZZ,N8,N,14.006700,19.123,30.040,13.602,267.850124,420.761268,190.519133
209,6ZIR,EZZ,O10,O,15.999400,19.664,27.911,12.947,314.612202,446.559253,207.144232


# Calculate mass centers of each ligand through all pdb files

In [108]:
df['wx'] = df['x'] * df['weight']
df['wy'] = df['y'] * df['weight']
df['wz'] = df['z'] * df['weight']


res = df.groupby(['protein', 'resname']).agg({
    'weight': 'sum',
    'wx': 'sum',
    'wy': 'sum',
    'wz': 'sum'
})

df_com = pd.DataFrame()
df_com['mass_x'] = res['wx'] / res['weight']
df_com['mass_y'] = res['wy'] / res['weight']
df_com['mass_z'] = res['wz'] / res['weight']

df_com = df_com.reset_index()

In [130]:
def find_nearby_residues(df_com, target_protein='6GJ6', target_res='EZZ', cutoff=5.0):
    """
    Returns all residues in the same protein within a specified distance
    of the target residue's center of mass.
    """
    # 1. Locate the target residue (EZZ) center of mass
    target_row = df_com[(df_com['protein'] == target_protein) & (df_com['resname'] == target_res)]
    if target_row.empty:
        return f"Target {target_res} not found in protein {target_protein}."
    target_xyz = target_row[['mass_x', 'mass_y', 'mass_z']].values[0]
    mask = ~((df_com['protein'] == target_protein) & (df_com['resname'] == target_res))
    other_ligands = df_com[mask].copy()

    # 3. Vectorized Euclidean distance calculation
    coords = other_ligands[['mass_x', 'mass_y', 'mass_z']].values
    other_ligands['distance_A'] = np.linalg.norm(coords - target_xyz, axis=1)

    nearby_df = other_ligands[other_ligands['distance_A'] <= cutoff].sort_values('distance_A').reset_index(drop=True)

    return nearby_df

In [134]:
ligands_bindingsite1 = find_nearby_residues(df_com, target_protein='6GJ6', target_res='EZZ', cutoff=6.0)
ligands_bindingsite1

,protein,resname,mass_x,mass_y,mass_z,distance_A
0,6ZJ0,EZZ,18.234468,31.339617,12.666162,0.382062
1,6ZL3,EZZ,18.528030,31.735212,12.108820,0.730639
2,6GJ7,F0B,19.385377,33.356253,13.860468,2.259607
3,6ZIZ,EZZ,21.482327,34.106939,15.690154,4.942240


In [135]:
ligands_bindingsite1.to_csv('ligands_bindingsite1.cvs', encoding='utf-8', index=False)

In [126]:
# 6ZIR contains 2 ligands in the same chain A

In [136]:
structures_dir = Path('./structures')
ligands_dir = Path('./ligands')

In [143]:
ligands_dictionary = ligands_bindingsite1.set_index('protein')['resname'].to_dict()
ligands_dictionary['6GJ6'] = 'EZZ'

In [170]:
ligands_dictionary

{'6ZJ0': 'EZZ', '6ZL3': 'EZZ', '6GJ7': 'F0B', '6ZIZ': 'EZZ', '6GJ6': 'EZZ'}

In [171]:
for cif_file in structures_dir.glob("*_out:A.cif"): # out:A.cif - aligned proteins chain A
    try:
        protein = gemmi.read_structure(str(cif_file.resolve()))
        mon_names.update(protein[0].get_all_residue_names())
        for model in protein:
            for chain in model:
                for residue in chain:
                    if ligands_dictionary.get(protein.name) == residue.name:
                        temp_st = gemmi.Structure()
                        temp_st.add_model(gemmi.Model("1"))
                        temp_st[0].add_chain(gemmi.Chain(chain.name))
                        temp_st[0][0].add_residue(residue, 0)
                        temp_pdb = f"./ligands/{protein.name}_{residue.name}.pdb"
                        temp_st.write_pdb(temp_pdb)

                        print(residue.name, protein.name, residue.entity_type)
    except Exception as e:
        print(f"Error processing {cif_file.name}: {e}")

EZZ 6GJ6 EntityType.NonPolymer
F0B 6GJ7 EntityType.NonPolymer
EZZ 6ZIZ EntityType.NonPolymer
EZZ 6ZIZ EntityType.NonPolymer
EZZ 6ZJ0 EntityType.NonPolymer
EZZ 6ZL3 EntityType.NonPolymer


In [172]:
from openbabel import pybel
import os

In [173]:
data_list = []


for pdb_file in ligands_dir.glob("*.pdb"):
    try:
        mol = next(pybel.readfile("pdb", str(pdb_file)))
        smiles = mol.write("smi").split()[0]
        prot_id = pdb_file.stem.split('_')[0]
        data_list.append({
            'protein': prot_id,
            'resname': pdb_file.stem.split('_')[1],
            'smiles': smiles
        })
    except Exception as e:
        print(f"Failed to read {pdb_file}: {e}")

df_ligands = pd.DataFrame(data_list)


*** Open Babel Warning  in PerceiveBondOrders
  Failed to kekulize aromatic bonds in OBMol::PerceiveBondOrders (title is ligands/6GJ6_EZZ.pdb)

*** Open Babel Warning  in PerceiveBondOrders
  Failed to kekulize aromatic bonds in OBMol::PerceiveBondOrders (title is ligands/6GJ7_F0B.pdb)

*** Open Babel Warning  in PerceiveBondOrders
  Failed to kekulize aromatic bonds in OBMol::PerceiveBondOrders (title is ligands/6ZL3_EZZ.pdb)



### List of all ligands in the first binding site within proximity of 6 Angstrem after alignment

In [155]:
df_ligands

,protein,resname,smiles
0,6GJ7,F0B,c1c(cc2[C@H](C3=c4c(=N[C@@H]3CNCc3ccc5c(c3)n(c...
1,6ZIZ,EZZ,c1c2c(ccc1O)C(=O)N[C@H]2c1c2c(cccc2)[nH]c1CN(C)C
2,6ZJ0,EZZ,c1c2c(ccc1O)C(=O)N[C@H]2c1c2c(cccc2)[nH]c1CN(C)C
3,6ZL3,EZZ,c1c2c(ccc1O)C(=O)N[C@H]2[C@@H]1c2c(cccc2)N=C1C...


### Critical: generates minimized conformers of cristallized ligands "from scratch" (smiles).

In [175]:
def smiles_to_sdf(smiles, sdf_path, pH=7.4):
    """
    Convert a SMILES string to a PDBQT file needed by docking programs of the AutoDock family.

    Parameters
    ----------
    smiles: str
        SMILES string.
    sdf_path: str or pathlib.path
        Path to output sdf file.
    pH: float
        Protonation at given pH.
    """
    molecule = pybel.readstring("smi", smiles)
    molecule.addh()
    molecule.OBMol.CorrectForPH(pH)
    molecule.make3D(forcefield="mmff94", steps=10000)
    molecule.localopt(forcefield="mmff94")
    # add partial charges to each atom
    #for atom in molecule.atoms:
    #    atom.OBAtom.GetPartialCharge()
    molecule.write("sdf", str(sdf_path), overwrite=True)
    return

In [176]:
ligands_dir = Path('./ligands')

In [177]:
for index, row in df_ligands.iterrows():
    smile = row["smiles"]
    pdb = row["protein"]
    ligand = row["resname"]
    sdf_path = ligands_dir / f"{pdb}_{ligand}.sdf"
    smiles_to_sdf(smile, str(sdf_path.resolve()), pH=7.4)

## Part of example of docking commands

### preparing of proteins as pdbqt files with partial charges didn't show any improvements in RMSD and docking binding energy (minimizedAffinity) compared to raw pdb files.
### Preparing of proteins as cell for MD-simulation (with AmberTools in Amber FF wasn't possible due to resources.

In [186]:
# conda install bioconda::mgltools
# smina -r 6GJ6.pdbqt -l 6GJ6_EZZ.sdf --autobox_ligand 6GJ6_EZZ.pdb --size_x 20 --size_y 20 --size_z 20 --cpu 4 --exhaustiveness 8 --seed -230353547 -o 6GJZ_redock.sdf
# prepare_receptor4.py -r 6GJ6.pdb -o 6GJ6.pdbqt -A checkhydrogens -U # nphs_lps_waters -e
# grep -v "EZZ" 6GJ6.pdb > 6GJ6_apo.pdb
# smina -r structures/6ZIZ_apo.pdb -l ligands/6ZIZ_EZZ.sdf --autobox_ligand ligands/6ZIZ_EZZ.pdb --size_x 20 --size_y 20 --size_z 20 --cpu 4 --exhaustiveness 8 --seed -230353547 -o docking/6ZIZ_redock.sdf

PosixPath('ligands/6ZL3_EZZ_protonated.pdb')

#### Work with cif files: possible automatical detection of connection and type of bonds (metal/covalent/hydrogen etc.) (it only for interest)

In [14]:
file_path = Path('./structures/raw/6GJ6.cif')

In [26]:
doc = gemmi.cif.read_file(str(file_path.resolve()))
block = doc[0]

target_tags = [
    '_struct_conn.id',
    '_struct_conn.conn_type_id',
    '_struct_conn.ptnr1_label_atom_id',
    '_struct_conn.ptnr2_label_atom_id',
    '_struct_conn.pdbx_dist_value' # This is the distance tag in your snippet
]

table = block.find(target_tags)

connections = pd.DataFrame(list(table), columns=table.tags)

In [27]:
connections

,_struct_conn.id,_struct_conn.conn_type_id,_struct_conn.ptnr1_label_atom_id,_struct_conn.ptnr2_label_atom_id,_struct_conn.pdbx_dist_value
0,metalc1,metalc,OG,MG,2.084
1,metalc2,metalc,OG1,MG,2.117
2,metalc3,metalc,MG,O2G,2.058
3,metalc4,metalc,MG,O2B,2.034
4,metalc5,metalc,MG,O,1.949
5,metalc6,metalc,MG,O,2.015


In [17]:
for block in doc:
    print(f"--- Data Block: {block.name} ---")

--- Data Block: 6GJ6 ---
